# NF-v3 preparation and relabelling

This notebook never modifies the raw CSV. It creates new artefacts in `nids-fair-retrain` and runs the repository's versioned script. Before running it, adjust only the paths in the next cell.

In [1]:

# ============================================================
# SETUP - Run this cell first
# ============================================================
!git clone -b feat/fair-retrain-clean https://github.com/tatipar/temporalgnn-nids.git
import sys
sys.path.append('/content/temporalgnn-nids/code/python')

# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/nids-mitre/')


Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 1203, done.
remote: Counting objects: 100% (412/412), done.
remote: Compressing objects: 100% (267/267), done.
remote: Total 1203 (delta 189), reused 205 (delta 81), pack-reused 791 (from 1)
Receiving objects: 100% (1203/1203), 10.29 MiB | 19.88 MiB/s, done.
Resolving deltas: 100% (449/449), done.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# NEW folder: corrected data, manifests, graphs, and results only.
PROJECT_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
# Repository cloned by this Colab session.
REPO_ROOT = Path('/content/temporalgnn-nids')
# Existing raw CSV: never written or moved.
# Existing day-level extracts. They are the preferred inputs when they preserve
# every original NF-v3 row and column for 28 February and 1 March.
INPUT_CSVS = [
    Path('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv'),
    Path('/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_thu0103.csv'),
]
OUTPUT_DIR = PROJECT_ROOT / 'corrected_data' / 'infiltration_v1'

for directory in [PROJECT_ROOT, OUTPUT_DIR, PROJECT_ROOT / 'graphs', PROJECT_ROOT / 'results']:
    directory.mkdir(parents=True, exist_ok=True)

assert REPO_ROOT.is_dir(), f'Repository not found: {REPO_ROOT}'
for path in INPUT_CSVS:
    assert path.is_file(), f'Raw CSV not found: {path}'

print(f'Project: {PROJECT_ROOT}')
print(f'Output: {OUTPUT_DIR}')
print('Inputs:', *INPUT_CSVS, sep='\n- ')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/nids-fair-retrain
Output: /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1
Inputs:
- /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv
- /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_thu0103.csv


## Optional: validate the Category-0 direction mapping

Run this section only if you have a CICFlowMeter CSV that retains source/destination IPs and ports. A CSV containing only aggregate features and timestamps cannot establish a trustworthy direction mapping. The cell compares `Total Length of Fwd Packets` against NF-v3 `IN_BYTES` and `OUT_BYTES` for matched Dropbox and victim-attacker flows.

In [4]:
# Set this to a CICFlowMeter CSV with endpoint columns. Leave as None to skip validation.
CIC_CSV = None  # Path('/content/drive/MyDrive/nids-mitre/data/cicids2018/...csv')

# Change values only if your CICFlowMeter export uses different column names.
CIC_COLUMNS = {
    'time': 'Timestamp',
    'source_ip': 'Src IP',
    'destination_ip': 'Dst IP',
    'source_port': 'Src Port',
    'destination_port': 'Dst Port',
    'forward_bytes': 'Total Length of Fwd Packets',
}
MATCH_TOLERANCE = '2s'
# Use the timezone represented by a naive CICFlowMeter Timestamp.
CIC_TIMESTAMP_TIMEZONE = 'UTC'


In [ ]:
import pandas as pd

if CIC_CSV is None:
    print('Set CIC_CSV to run the direction-mapping validation.')
else:
    CIC_CSV = Path(CIC_CSV)
    assert CIC_CSV.is_file(), f'CICFlowMeter CSV not found: {CIC_CSV}'

    # CICFlowMeter must retain these keys; timestamp-only matching is ambiguous.
    cic_header = pd.read_csv(CIC_CSV, nrows=0).columns.str.strip().tolist()
    missing = [column for column in CIC_COLUMNS.values() if column not in cic_header]
    if missing:
        raise ValueError(
            f'CICFlowMeter CSV cannot validate the direction mapping; missing columns: {missing}. '
            'Use an export with endpoint IPs/ports or validate against PCAP-derived flows.'
        )

    # Category 0 is relevant only to these documented endpoints.
    victims = {'172.31.69.24', '172.31.69.13'}
    category_0_destinations = {
        '162.125.3.1', '162.125.3.5', '162.125.3.6', '162.125.248.1',
        '162.125.18.133', '13.58.225.34',
    }

    nf_columns = [
        'FLOW_START_MILLISECONDS', 'IPV4_SRC_ADDR', 'IPV4_DST_ADDR',
        'L4_SRC_PORT', 'L4_DST_PORT', 'IN_BYTES', 'OUT_BYTES',
    ]
    nf_parts = []
    # for nf_chunk in pd.read_csv(INPUT_CSVS[0], usecols=nf_columns, chunksize=500_000):
    #     candidate = (
    #         nf_chunk['IPV4_SRC_ADDR'].astype(str).isin(victims)
    #         & nf_chunk['IPV4_DST_ADDR'].astype(str).isin(category_0_destinations)
    #     )
    #     nf_parts.append(nf_chunk.loc[candidate])
    for input_csv in INPUT_CSVS:
        for nf_chunk in pd.read_csv(input_csv, usecols=nf_columns, chunksize=500_000):
            candidate = (
                nf_chunk['IPV4_SRC_ADDR'].astype(str).isin(victims)
                & nf_chunk['IPV4_DST_ADDR'].astype(str).isin(category_0_destinations)
            )
            nf_parts.append(nf_chunk.loc[candidate])

    nf = pd.concat(nf_parts, ignore_index=True)
    nf['match_time'] = pd.to_datetime(nf['FLOW_START_MILLISECONDS'], unit='ms', utc=True)
    nf = nf.rename(columns={
        'IPV4_SRC_ADDR': 'source_ip', 'IPV4_DST_ADDR': 'destination_ip',
        'L4_SRC_PORT': 'source_port', 'L4_DST_PORT': 'destination_port',
    })

    cic_usecols = list(CIC_COLUMNS.values())
    cic_parts = []
    for cic_chunk in pd.read_csv(CIC_CSV, usecols=cic_usecols, chunksize=500_000):
        cic_chunk.columns = cic_chunk.columns.str.strip()
        candidate = (
            cic_chunk[CIC_COLUMNS['source_ip']].astype(str).isin(victims)
            & cic_chunk[CIC_COLUMNS['destination_ip']].astype(str).isin(category_0_destinations)
        )
        cic_parts.append(cic_chunk.loc[candidate])
    cic = pd.concat(cic_parts, ignore_index=True).rename(columns={value: key for key, value in CIC_COLUMNS.items()})
    cic_time = pd.to_datetime(cic['time'], errors='coerce')
    if getattr(cic_time.dt, 'tz', None) is None:
        cic_time = cic_time.dt.tz_localize(CIC_TIMESTAMP_TIMEZONE)
    cic['match_time'] = cic_time.dt.tz_convert('UTC')
    cic = cic.dropna(subset=['match_time'])

    match_keys = ['source_ip', 'destination_ip', 'source_port', 'destination_port']
    for key in ['source_port', 'destination_port']:
        nf[key] = pd.to_numeric(nf[key], errors='coerce').astype('Int64')
        cic[key] = pd.to_numeric(cic[key], errors='coerce').astype('Int64')

    matched = pd.merge_asof(
        nf.sort_values('match_time'), cic.sort_values('match_time'),
        on='match_time', by=match_keys, direction='nearest',
        tolerance=pd.Timedelta(MATCH_TOLERANCE), suffixes=('_nf', '_cic'),
    ).dropna(subset=['forward_bytes'])

    if matched.empty:
        raise ValueError('No flows matched. Check timestamp timezone, endpoint columns, and MATCH_TOLERANCE.')

    forward = pd.to_numeric(matched['forward_bytes'], errors='coerce')
    incoming = pd.to_numeric(matched['IN_BYTES'], errors='coerce')
    outgoing = pd.to_numeric(matched['OUT_BYTES'], errors='coerce')
    summary = pd.DataFrame({
        'exact_byte_agreement': [(forward == incoming).mean(), (forward == outgoing).mean()],
        'zero_indicator_agreement': [(forward.eq(0) == incoming.eq(0)).mean(), (forward.eq(0) == outgoing.eq(0)).mean()],
    }, index=['IN_BYTES', 'OUT_BYTES'])
    print(f'Matched flows: {len(matched):,}')
    display(summary.sort_values('zero_indicator_agreement', ascending=False))
    print('\nZero/non-zero agreement with Total Length of Fwd Packets:')
    display(pd.crosstab(forward.eq(0), incoming.eq(0), rownames=['Fwd bytes == 0'], colnames=['IN_BYTES == 0']))
    display(pd.crosstab(forward.eq(0), outgoing.eq(0), rownames=['Fwd bytes == 0'], colnames=['OUT_BYTES == 0']))
    print(
        '\nUse FORWARD_BYTES_COLUMN only if one candidate has near-perfect zero-indicator agreement '
        'and the result is stable when MATCH_TOLERANCE is changed (for example, 1s and 3s).'
    )


## Run

Category-4 `Attempted` flows become benign.

Category 0: With the [NF-v3 documentation](https://arxiv.org/abs/2503.04404), we do have a reasonable basis for choosing:

`FORWARD_BYTES_COLUMN` = `IN_BYTES`

The evidence is that the specification itself associates IN metrics with src → dst, for example:

`RETRANSMITTED IN BYTES: src->dst`

and that corresponds to the forward direction used in the rule, given that `IPV4_SRC_ADDR` is the source and `IPV4_DST_ADDR` is the destination.

There is a caveat: `Total Length of Forwarded Packets` from CICFlowMeter and `IN_BYTES` from NF-v3 can differ
in how they count headers. Therefore, there might not be an exact byte equality; but for the condition
that matters to us, == 0, `IN_BYTES` is the correct and documentable directional proxy.

In [6]:
import subprocess

FORWARD_BYTES_COLUMN = 'IN_BYTES'
command = [
    'python', str(REPO_ROOT / 'code/python/scripts/prepare_nfv3.py'),
    '--output-dir', str(OUTPUT_DIR),
    '--chunksize', '500000',
]
for input_csv in INPUT_CSVS:
    command += ['--input-csv', str(input_csv)]
if FORWARD_BYTES_COLUMN:
    command += ['--forward-bytes-column', FORWARD_BYTES_COLUMN]

print(' '.join(command))
subprocess.run(command, check=True)

python /content/temporalgnn-nids/code/python/scripts/prepare_nfv3.py --output-dir /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1 --chunksize 500000 --input-csv /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv --input-csv /content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_thu0103.csv --forward-bytes-column IN_BYTES


CompletedProcess(args=['python', '/content/temporalgnn-nids/code/python/scripts/prepare_nfv3.py', '--output-dir', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1', '--chunksize', '500000', '--input-csv', '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_wed2802.csv', '--input-csv', '/content/drive/MyDrive/nids-mitre/data/cicids2018-v3/cicids2018v3_thu0103.csv', '--forward-bytes-column', 'IN_BYTES'], returncode=0)

## Audit and manual review

Run this after preparation. It verifies the output hash, manifest counts, contiguous source-row provenance, valid binary targets, and a complete replay of the rules. It also creates a deterministic sample CSV with rows from every rule. Review the samples against the documentation before freezing this dataset version.


In [18]:
CORRECTED_CSV = OUTPUT_DIR / 'nfv3_corrected.csv'
MANIFEST_PATH = OUTPUT_DIR / 'nfv3_corrected.manifest.json'
AUDIT_DIR = OUTPUT_DIR / 'audit'

audit_command = [
    'python', str(REPO_ROOT / 'code/python/scripts/audit_nfv3_relabel.py'),
    '--corrected-csv', str(CORRECTED_CSV),
    '--manifest', str(MANIFEST_PATH),
    '--output-dir', str(AUDIT_DIR),
    '--sample-per-rule', '10',
]

if FORWARD_BYTES_COLUMN:
    audit_command += ['--forward-bytes-column', FORWARD_BYTES_COLUMN]
# Diagnostic only: compare the alternative direction without rewriting the CSV
audit_command += ['--counterfactual-forward-bytes-column', 'OUT_BYTES']

subprocess.run(audit_command, check=True)
print('Manual-review CSV:', AUDIT_DIR / 'nfv3_corrected.manual_review_samples.csv')

Manual-review CSV: /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/audit/nfv3_corrected.manual_review_samples.csv


In [19]:
# Inspect the small review file directly in Colab
review = pd.read_csv(AUDIT_DIR / 'nfv3_corrected.manual_review_samples.csv')
review.groupby('correction_rule').size(), review.head()

(correction_rule
 attempted_category_4_to_benign             10
 confirmed_dropbox_download                 10
 confirmed_nmap_portscan                    10
 confirmed_victim_attacker_communication    10
 old_infilteration_to_benign                10
 unchanged                                  10
 dtype: int64,
    source_row_id               source_file  FLOW_START_MILLISECONDS  \
 0        2446559  cicids2018v3_thu0103.csv            1519912394877   
 1        2457043  cicids2018v3_thu0103.csv            1519912529294   
 2         501588  cicids2018v3_wed2802.csv            1519828407231   
 3        2458000  cicids2018v3_thu0103.csv            1519912542372   
 4        2446514  cicids2018v3_thu0103.csv            1519912393791   
 
    FLOW_END_MILLISECONDS IPV4_SRC_ADDR  L4_SRC_PORT  IPV4_DST_ADDR  \
 0          1519912513294  172.31.69.13        50851  13.32.168.125   
 1          1519912583640  172.31.69.13        50836  104.16.100.29   
 2          1519828477415  172.31.69.24

In [20]:
columns = [
    "source_file", "FLOW_START_TIME",
    "IPV4_SRC_ADDR", "L4_SRC_PORT",
    "IPV4_DST_ADDR", "L4_DST_PORT",
    "Attack", "binary_target",
    "correction_rule", "attempted_category",
]

for rule, rows in review.groupby("correction_rule"):
    print(f"\n--- {rule} ---")
    display(rows[columns].sort_values("FLOW_START_TIME"))



--- attempted_category_4_to_benign ---


,source_file,FLOW_START_TIME,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,Attack,binary_target,correction_rule,attempted_category
9,cicids2018v3_wed2802.csv,2018-02-28 14:33:25.142,172.31.69.24,51361,104.16.100.29,443,Benign,0,attempted_category_4_to_benign,4
7,cicids2018v3_wed2802.csv,2018-02-28 14:33:25.856,172.31.69.24,51364,104.16.100.29,443,Benign,0,attempted_category_4_to_benign,4
2,cicids2018v3_wed2802.csv,2018-02-28 14:33:27.231,172.31.69.24,51371,52.85.101.236,443,Benign,0,attempted_category_4_to_benign,4
6,cicids2018v3_wed2802.csv,2018-02-28 14:33:28.035,172.31.69.24,51375,52.85.131.81,443,Benign,0,attempted_category_4_to_benign,4
8,cicids2018v3_thu0103.csv,2018-03-01 13:53:11.836,172.31.69.13,50840,104.16.100.29,443,Benign,0,attempted_category_4_to_benign,4
4,cicids2018v3_thu0103.csv,2018-03-01 13:53:13.791,172.31.69.13,50849,52.85.112.72,443,Benign,0,attempted_category_4_to_benign,4
0,cicids2018v3_thu0103.csv,2018-03-01 13:53:14.877,172.31.69.13,50851,13.32.168.125,443,Benign,0,attempted_category_4_to_benign,4
1,cicids2018v3_thu0103.csv,2018-03-01 13:55:29.294,172.31.69.13,50836,104.16.100.29,443,Benign,0,attempted_category_4_to_benign,4
3,cicids2018v3_thu0103.csv,2018-03-01 13:55:42.372,172.31.69.13,50835,104.16.100.29,443,Benign,0,attempted_category_4_to_benign,4
5,cicids2018v3_thu0103.csv,2018-03-01 13:58:22.135,172.31.69.13,50893,104.16.100.29,443,Infilteration,0,attempted_category_4_to_benign,4



--- confirmed_dropbox_download ---


,source_file,FLOW_START_TIME,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,Attack,binary_target,correction_rule,attempted_category
14,cicids2018v3_wed2802.csv,2018-02-28 14:33:26.012,172.31.69.24,51369,162.125.3.1,443,Benign,1,confirmed_dropbox_download,-1
19,cicids2018v3_thu0103.csv,2018-03-01 13:53:11.237,172.31.69.13,50834,162.125.3.1,443,Benign,1,confirmed_dropbox_download,-1
17,cicids2018v3_thu0103.csv,2018-03-01 13:53:12.606,172.31.69.13,50844,162.125.3.1,443,Benign,1,confirmed_dropbox_download,-1
10,cicids2018v3_thu0103.csv,2018-03-01 13:56:25.652,172.31.69.13,50856,162.125.18.133,443,Benign,1,confirmed_dropbox_download,-1
11,cicids2018v3_thu0103.csv,2018-03-01 14:05:55.119,172.31.69.13,50855,162.125.18.133,443,Infilteration,1,confirmed_dropbox_download,-1
13,cicids2018v3_thu0103.csv,2018-03-01 14:07:57.109,172.31.69.13,50855,162.125.18.133,443,Infilteration,1,confirmed_dropbox_download,-1
16,cicids2018v3_thu0103.csv,2018-03-01 14:14:06.317,172.31.69.13,50855,162.125.18.133,443,Infilteration,1,confirmed_dropbox_download,-1
12,cicids2018v3_thu0103.csv,2018-03-01 14:16:09.422,172.31.69.13,50855,162.125.18.133,443,Infilteration,1,confirmed_dropbox_download,-1
15,cicids2018v3_thu0103.csv,2018-03-01 14:24:59.025,172.31.69.13,51393,162.125.18.133,443,Infilteration,1,confirmed_dropbox_download,-1
18,cicids2018v3_thu0103.csv,2018-03-01 14:30:30.847,172.31.69.13,50855,162.125.18.133,443,Infilteration,1,confirmed_dropbox_download,-1



--- confirmed_nmap_portscan ---


...

,source_file,FLOW_START_TIME,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,Attack,binary_target,correction_rule,attempted_category
32,cicids2018v3_wed2802.csv,2018-02-28 14:51:20.673,172.31.69.24,51603,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
35,cicids2018v3_wed2802.csv,2018-02-28 15:00:37.963,172.31.69.24,51603,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
36,cicids2018v3_wed2802.csv,2018-02-28 15:09:08.150,172.31.69.24,51603,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
34,cicids2018v3_wed2802.csv,2018-02-28 16:03:25.562,172.31.69.24,51603,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
38,cicids2018v3_wed2802.csv,2018-02-28 16:04:28.248,172.31.69.24,51603,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
39,cicids2018v3_wed2802.csv,2018-02-28 18:35:22.661,172.31.69.24,54751,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
33,cicids2018v3_wed2802.csv,2018-02-28 18:36:29.349,172.31.69.24,54751,13.58.225.34,31337,Benign,1,confirmed_victim_attacker_communication,-1
30,cicids2018v3_thu0103.csv,2018-03-01 13:57:54.262,172.31.69.13,50887,13.58.225.34,31337,Infilteration,1,confirmed_victim_attacker_communication,-1
31,cicids2018v3_thu0103.csv,2018-03-01 14:04:35.178,172.31.69.13,51040,13.58.225.34,31337,Infilteration,1,confirmed_victim_attacker_communication,-1
37,cicids2018v3_thu0103.csv,2018-03-01 14:16:43.046,172.31.69.13,51040,13.58.225.34,31337,Infilteration,1,confirmed_victim_attacker_communication,-1



--- old_infilteration_to_benign ---


,source_file,FLOW_START_TIME,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,Attack,binary_target,correction_rule,attempted_category
48,cicids2018v3_wed2802.csv,2018-02-28 13:46:37.101,172.31.69.16,60787,172.31.0.2,53,Infilteration,0,old_infilteration_to_benign,-1
45,cicids2018v3_wed2802.csv,2018-02-28 15:17:37.040,172.31.69.9,57379,172.31.0.2,53,Infilteration,0,old_infilteration_to_benign,-1
49,cicids2018v3_wed2802.csv,2018-02-28 15:28:53.410,172.31.69.13,58197,172.31.0.2,53,Infilteration,0,old_infilteration_to_benign,-1
47,cicids2018v3_wed2802.csv,2018-02-28 18:15:49.371,172.31.69.17,59082,172.31.0.2,53,Infilteration,0,old_infilteration_to_benign,-1
46,cicids2018v3_wed2802.csv,2018-02-28 18:25:39.616,172.31.69.10,55552,172.217.7.238,443,Infilteration,0,old_infilteration_to_benign,-1
40,cicids2018v3_wed2802.csv,2018-02-28 18:37:09.432,172.31.69.7,45719,91.189.91.157,123,Infilteration,0,old_infilteration_to_benign,-1
42,cicids2018v3_wed2802.csv,2018-02-28 20:00:21.775,172.31.69.5,50999,172.31.0.2,53,Infilteration,0,old_infilteration_to_benign,-1
41,cicids2018v3_thu0103.csv,2018-03-01 17:50:03.252,172.31.69.5,58926,172.31.0.2,53,Infilteration,0,old_infilteration_to_benign,-1
43,cicids2018v3_thu0103.csv,2018-03-01 18:52:45.908,5.101.40.105,60885,172.31.69.14,3389,Infilteration,0,old_infilteration_to_benign,-1
44,cicids2018v3_thu0103.csv,2018-03-01 21:18:22.815,172.31.69.16,123,13.89.190.88,123,Infilteration,0,old_infilteration_to_benign,-1



--- unchanged ---


,source_file,FLOW_START_TIME,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,Attack,binary_target,correction_rule,attempted_category
58,cicids2018v3_wed2802.csv,2018-02-28 13:52:03.780,172.31.66.73,3389,193.111.198.70,59738,Benign,0,unchanged,-1
59,cicids2018v3_wed2802.csv,2018-02-28 15:28:57.413,172.31.66.43,3389,213.202.230.143,2625,Benign,0,unchanged,-1
55,cicids2018v3_wed2802.csv,2018-02-28 15:37:59.330,172.31.68.4,3389,193.111.198.69,43355,Benign,0,unchanged,-1
56,cicids2018v3_wed2802.csv,2018-02-28 18:14:22.304,172.31.67.23,52467,172.31.0.2,53,Benign,0,unchanged,-1
57,cicids2018v3_wed2802.csv,2018-02-28 18:22:35.157,172.31.69.26,54591,216.105.38.9,443,Benign,0,unchanged,-1
50,cicids2018v3_wed2802.csv,2018-02-28 18:34:34.652,173.208.136.146,57950,172.31.66.100,3389,Benign,0,unchanged,-1
52,cicids2018v3_wed2802.csv,2018-02-28 19:51:12.713,222.114.6.130,55741,172.31.64.24,3389,Benign,0,unchanged,-1
51,cicids2018v3_thu0103.csv,2018-03-01 17:49:58.081,5.101.40.43,56929,172.31.66.11,3389,Benign,0,unchanged,-1
53,cicids2018v3_thu0103.csv,2018-03-01 19:06:24.675,172.31.66.25,63053,172.31.0.2,53,Benign,0,unchanged,-1
54,cicids2018v3_thu0103.csv,2018-03-01 21:02:47.954,172.31.66.90,51731,52.218.193.202,443,Benign,0,unchanged,-1


In [21]:
import json
from pathlib import Path

audit_path = Path(
    "/content/drive/MyDrive/nids-fair-retrain/"
    "corrected_data/infiltration_v1/audit/"
    "nfv3_corrected.audit.json"
)

with audit_path.open() as file:
    audit = json.load(file)


print("Status:", audit["status"])
print("Checks:", audit["checks"])
print("Binary transitions:", audit["binary_transitions"])
print("Changed labels (%):", audit["changed_binary_label_pct"])
print("Category 0:", audit["category_0_detection"])
print("OUT_BYTES counterfactual:", audit["category_0_counterfactuals"])


Status: passed
Checks: {'row_count_matches_manifest': True, 'sha256_matches_manifest': True, 'binary_counts_match_manifest': True, 'rule_counts_match_manifest': True}
Binary transitions: {'0->0': 3931453, '0->1': 60655, '1->0': 136755, '1->1': 51397}
Changed labels (%): 4.722433532842455
Category 0: enabled using IN_BYTES
OUT_BYTES counterfactual: {'OUT_BYTES': {'attempted_category_0_rows': 0, 'binary_target_changes_vs_stored': 0}}


In [22]:
import json

manifest_path = OUTPUT_DIR / 'nfv3_corrected.manifest.json'
manifest = json.loads(manifest_path.read_text())
print(json.dumps(manifest['counts'], indent=2))
print('\nCategory 0:', manifest['category_0_detection'])

{
  "original_attack_labels": {
    "Benign": 3992108,
    "Infilteration": 188152
  },
  "corrected_detail_labels": {
    "Benign": 4068208,
    "Infiltration - Communication Victim Attacker": 12,
    "Infiltration - Dropbox Download": 48,
    "Infiltration - NMAP Portscan": 111992
  },
  "correction_rules": {
    "attempted_category_4_to_benign": 18,
    "confirmed_dropbox_download": 48,
    "confirmed_nmap_portscan": 111992,
    "confirmed_victim_attacker_communication": 12,
    "old_infilteration_to_benign": 136754,
    "unchanged": 3931436
  },
  "binary_before": {
    "0": 3992108,
    "1": 188152
  },
  "binary_after": {
    "0": 4068208,
    "1": 112052
  }
}

Category 0: enabled using IN_BYTES
